# EGNN Dataset Preparation Pipeline

Complete end-to-end pipeline:
1. Load KLIFS dataset
2. Clean & validate data
3. Define unique samples
4. Extract Cα structures (graphs)
5. Build ESM embeddings

Output: Ready for EGNN/XGBoost training

In [1]:
import subprocess
import sys

packages = [
    ("pandas", "pandas"),
    ("biopython", "Bio"),
    ("torch", "torch"),
    ("fair-esm", "esm")
]

for pip_name, import_name in packages:
    try:
        __import__(import_name)
    except ImportError:
        print(f"Installing {pip_name}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])

Installing fair-esm...


In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
import json
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

# Biopython for PDB parsing
from Bio import PDB
from Bio.PDB import PDBParser, PDBIO

# PyTorch for graph operations and ESM
import torch
import torch.nn.functional as F

# ESM for embeddings
try:
    import esm
except ImportError:
    print("Warning: ESM not installed. Will install on demand.")

print("✓ All imports successful")

✓ All imports successful


## 1. Load KLIFS Dataset

In [3]:
BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "data"
CSV_FILE = DATA_DIR / "all_structures.csv"
PDB_DIR = DATA_DIR / "raw" / "pdbs"

# Load dataset
df = pd.read_csv(CSV_FILE)

print(f"✓ Loaded KLIFS dataset: {CSV_FILE}")
print(f"  Shape: {df.shape}")
print(f"  Columns: {list(df.columns)}")
print(f"\nFirst rows:")
print(df.head(3))
print(f"\nData types:")
print(df.dtypes)

✓ Loaded KLIFS dataset: /Users/josefinadehan/vision_avanzada_t/vision_avanzada_t/data/all_structures.csv
  Shape: (1267, 13)
  Columns: ['kinase_name', 'structure_id', 'pdb_id', 'dfg_state', 'alphac_state', 'conformation_class', 'resolution', 'species', 'chain', 'quality_score', 'ligand', 'pocket', 'klifs_residue_map']

First rows:
  kinase_name  structure_id pdb_id dfg_state alphac_state conformation_class  \
0        EGFR           782   3w33        in          out           INACTIVE   
1        EGFR           796   4ll0        in           in             ACTIVE   
2        EGFR           783   2ito        in           in             ACTIVE   

   resolution species chain  quality_score ligand  \
0        1.70   Human     A            8.0    W19   
1        4.00   Human     B            8.5    YUN   
2        3.25   Human     A            8.0    IRE   

                                              pocket  \
0  KVLGSGAFGTVYKVAIKELEILDEAYVMASVDPHVCRLLGIQLITQ...   
1  KVLGSGAFGTVYKVAIK

## 2. Clean Dataset

In [4]:
print("BEFORE CLEANING:")
print(f"  Total rows: {len(df)}")
print(f"  Missing values:\n{df.isnull().sum()}")

# Required columns for pipeline
REQUIRED_COLS = ['kinase_name', 'structure_id', 'pdb_id', 'dfg_state', 
                 'alphac_state', 'conformation_class', 'resolution', 'species', 'chain']
OPTIONAL_COLS = ['pocket', 'klifs_residue_map']

# Check if all required columns exist
missing_cols = set(REQUIRED_COLS) - set(df.columns)
if missing_cols:
    print(f"⚠ Missing columns: {missing_cols}")
    print(f"  Available: {list(df.columns)}")
else:
    print(f"✓ All required columns present")

# Remove rows with NaN in critical columns
df_clean = df.dropna(subset=['kinase_name', 'pdb_id', 'conformation_class'])

# Validate PDB files exist on disk
pdb_files_exist = []
missing_pdb_files = []
for idx, row in df_clean.iterrows():
    pdb_code = row['pdb_id'].lower()
    pdb_file = PDB_DIR / f"{pdb_code}.pdb"
    if pdb_file.exists():
        pdb_files_exist.append(True)
    else:
        pdb_files_exist.append(False)
        if len(missing_pdb_files) < 5:
            missing_pdb_files.append(pdb_code)

df_clean['pdb_file_exists'] = pdb_files_exist
pdb_exists_count = sum(pdb_files_exist)

print(f"\nAFTER CLEANING:")
print(f"  Total rows: {len(df_clean)}")
print(f"  PDB files found: {pdb_exists_count}/{len(df_clean)}")
if missing_pdb_files:
    print(f"  Sample missing files: {missing_pdb_files}")

# Remove entries without PDB files
df_clean = df_clean[df_clean['pdb_file_exists']].drop('pdb_file_exists', axis=1)
print(f"  After PDB validation: {len(df_clean)} rows")

print(f"\nConformation class distribution:")
print(df_clean['conformation_class'].value_counts())
print(f"\nKinase distribution:")
print(df_clean['kinase_name'].value_counts())

BEFORE CLEANING:
  Total rows: 1267
  Missing values:
kinase_name           0
structure_id          0
pdb_id                0
dfg_state             0
alphac_state          0
conformation_class    0
resolution            0
species               0
chain                 0
quality_score         0
ligand                0
pocket                0
klifs_residue_map     0
dtype: int64
✓ All required columns present

AFTER CLEANING:
  Total rows: 1267
  PDB files found: 1228/1267
  Sample missing files: ['4i1z', '4i1z', '4i21', '4i21', '3ue4']
  After PDB validation: 1228 rows

Conformation class distribution:
conformation_class
INACTIVE    662
ACTIVE      546
UNKNOWN      20
Name: count, dtype: int64

Kinase distribution:
kinase_name
EGFR      561
BRAF      243
FGFR1     217
ABL1      126
KIT        67
PDGFRA     14
Name: count, dtype: int64


## 3. Define Unique Samples (CRITICAL)

In [5]:
print("UNIQUE SAMPLE DEFINITION:")
print("  Identifier: kinase_name + pdb_id + chain_id")
print("  Logic: One structure per unique kinase-PDB-chain combination\n")

# Try to extract chain from structure_id or use default
# Most KLIFS entries store chain information; we'll use structure_id as proxy
# If chain info is available in data, use it; otherwise assume single chain

# Get chain info if available in dataframe
if 'chain' in df_clean.columns:
    print("✓ Chain information found in data")
    df_clean['sample_id'] = (df_clean['kinase_name'] + "_" + 
                             df_clean['pdb_id'].str.lower() + "_" + 
                             df_clean['chain'].astype(str))
else:
    # No chain info: use structure_id as differentiator
    print("⚠ Chain info not in data, using structure_id")
    df_clean['sample_id'] = (df_clean['kinase_name'] + "_" + 
                             df_clean['pdb_id'].str.lower() + "_" + 
                             df_clean['structure_id'].astype(str))

# Detect duplicates
duplicates = df_clean.duplicated(subset=['sample_id'], keep=False)
print(f"\nDuplicate sample IDs: {duplicates.sum()}")

if duplicates.sum() > 0:
    print("\nDuplicates found (keeping first occurrence):")
    dup_groups = df_clean[duplicates].groupby('sample_id').size()
    print(f"  Number of duplicate groups: {len(dup_groups)}")
    print(f"  Duplicates per group (sample): {dup_groups.head()}")
    
    # Keep first occurrence
    df_unique = df_clean.drop_duplicates(subset=['sample_id'], keep='first')
else:
    df_unique = df_clean.copy()

print(f"\nStatistics before deduplication: {len(df_clean)}")
print(f"Statistics after deduplication:  {len(df_unique)}")
print(f"Removed: {len(df_clean) - len(df_unique)} duplicate entries")

print(f"\nFinal conformation class distribution:")
print(df_unique['conformation_class'].value_counts())
print(f"\nFinal kinase distribution:")
print(df_unique['kinase_name'].value_counts())

UNIQUE SAMPLE DEFINITION:
  Identifier: kinase_name + pdb_id + chain_id
  Logic: One structure per unique kinase-PDB-chain combination

✓ Chain information found in data

Duplicate sample IDs: 803

Duplicates found (keeping first occurrence):
  Number of duplicate groups: 370
  Duplicates per group (sample): sample_id
ABL1_2gqg_A    2
ABL1_2gqg_B    2
ABL1_2hz0_A    2
ABL1_2hz0_B    2
ABL1_3cs9_A    2
dtype: int64

Statistics before deduplication: 1228
Statistics after deduplication:  795
Removed: 433 duplicate entries

Final conformation class distribution:
conformation_class
INACTIVE    424
ACTIVE      368
UNKNOWN       3
Name: count, dtype: int64

Final kinase distribution:
kinase_name
EGFR      373
BRAF      194
FGFR1     125
ABL1       49
KIT        44
PDGFRA     10
Name: count, dtype: int64


## 4. Extract Cα Structures (PDB → Graphs)

In [6]:
import json


def _clean_optional_value(value):
    if pd.isna(value):
        return None
    return value


def _parse_json_field(value):
    if value is None or pd.isna(value):
        return None
    if isinstance(value, (list, dict)):
        return value
    try:
        return json.loads(value)
    except Exception:
        return None


def _normalize_xray_position(value):
    if value is None or pd.isna(value):
        return None
    text = str(value).strip()
    if not text:
        return None
    # KLIFS suele devolver posiciones X-ray como "315" o "315A"; si viniera con cadena,
    # nos quedamos con el residuo para cruzarlo contra residue_ids del PDB parseado.
    if ':' in text:
        text = text.split(':', 1)[1]
    return text.strip()


def build_klifs_to_ca_index(klifs_residue_map, residue_ids):
    """Map KLIFS pocket position (1-85) to the row index in ca_coords when possible."""
    records = _parse_json_field(klifs_residue_map)
    if not records or not residue_ids:
        return None, 'klifs_residue_map_missing'

    residue_to_idx = {}
    for idx, residue_id in enumerate(residue_ids):
        residue_to_idx[str(residue_id)] = idx
        if ':' in str(residue_id):
            residue_to_idx[str(residue_id).split(':', 1)[1]] = idx

    mapping = {}
    for rec in records:
        if not isinstance(rec, dict):
            continue
        klifs_pos = rec.get('index') or rec.get('KLIFS_position') or rec.get('klifs_position')
        xray_pos = rec.get('Xray_position') or rec.get('xray_position')
        xray_key = _normalize_xray_position(xray_pos)
        try:
            klifs_key = int(klifs_pos)
        except Exception:
            continue
        if xray_key in residue_to_idx:
            mapping[klifs_key] = residue_to_idx[xray_key]

    if not mapping:
        return None, 'no_xray_positions_matched_ca_residue_ids'
    return mapping, f'mapped_{len(mapping)}_klifs_positions'


def extract_ca_coordinates(pdb_file, preferred_chain=None):
    """
    Extract Cα atom coordinates from a PDB file for one chain.
    
    Returns:
        dict with keys:
        - 'nodes': np.array of shape (N, 3) with Cα coordinates
        - 'residues': list of residue info
        - 'residue_ids': list aligned 1:1 with nodes, e.g. A:315
        - 'num_residues': N
        - 'chain_used': selected chain
        - 'valid': bool
    """
    try:
        parser = PDBParser(QUIET=True)
        structure = parser.get_structure('temp', str(pdb_file))

        ca_coords = []
        residues_info = []
        residue_ids = []
        selected_chain = str(preferred_chain).strip() if preferred_chain is not None and str(preferred_chain).strip() else None
        first_chain = None

        for model in structure:
            for chain in model:
                chain_id = chain.id.strip() or '_'
                if first_chain is None:
                    first_chain = chain_id
                if selected_chain is None:
                    selected_chain = first_chain
                if chain_id != selected_chain:
                    continue
                for residue in chain:
                    if residue.has_id('CA'):
                        ca_atom = residue['CA']
                        resseq = residue.get_id()[1]
                        icode = residue.get_id()[2].strip()
                        residue_id = f"{chain_id}:{resseq}{icode}"
                        ca_coords.append(ca_atom.get_coord())
                        residues_info.append({
                            'chain': chain_id,
                            'res_num': resseq,
                            'icode': icode,
                            'res_name': residue.get_resname(),
                            'residue_id': residue_id,
                        })
                        residue_ids.append(residue_id)

        if len(ca_coords) == 0:
            return {
                'nodes': np.array([]),
                'residues': [],
                'residue_ids': [],
                'num_residues': 0,
                'chain_used': selected_chain,
                'valid': False
            }

        return {
            'nodes': np.array(ca_coords, dtype=np.float32),
            'residues': residues_info,
            'residue_ids': residue_ids,
            'num_residues': len(ca_coords),
            'chain_used': selected_chain,
            'valid': True
        }
    except Exception as e:
        return {
            'nodes': np.array([]),
            'residues': [],
            'residue_ids': [],
            'num_residues': 0,
            'chain_used': None,
            'valid': False,
            'error': str(e)
        }

def build_distance_graph(ca_coords, distance_threshold=10.0):
    """
    Build adjacency matrix from Cα coordinates using distance threshold.
    
    Args:
        ca_coords: np.array of shape (N, 3)
        distance_threshold: Euclidean distance cutoff in Angstroms
    
    Returns:
        np.array of shape (N, N) with adjacency matrix
    """
    n_atoms = ca_coords.shape[0]
    
    # Compute pairwise distances
    diff = ca_coords[:, np.newaxis, :] - ca_coords[np.newaxis, :, :]  # (N, N, 3)
    distances = np.sqrt(np.sum(diff**2, axis=2))  # (N, N)
    
    # Build adjacency: connected if within threshold and not self
    adjacency = (distances < distance_threshold).astype(np.float32)
    np.fill_diagonal(adjacency, 0)  # No self-loops
    
    return adjacency

print("✓ Graph extraction functions defined\n")

# Extract graphs for all samples
structures_data = []
ca_extraction_errors = 0
invalid_structures = 0

print("Extracting Cα coordinates from PDB files...\n")

for idx, row in df_unique.iterrows():
    pdb_code = row['pdb_id'].lower()
    pdb_file = PDB_DIR / f"{pdb_code}.pdb"
    
    if not pdb_file.exists():
        ca_extraction_errors += 1
        continue
    
    # Extract Cα for the KLIFS/PDB chain associated with this row.
    ca_data = extract_ca_coordinates(pdb_file, preferred_chain=row.get('chain', None))
    
    if not ca_data['valid']:
        invalid_structures += 1
        continue
    
    # Build graph
    ca_coords = ca_data['nodes']
    adjacency = build_distance_graph(ca_coords, distance_threshold=10.0)
    num_edges = np.sum(adjacency) // 2  # Undirected graph
    klifs_to_ca_index, klifs_mapping_status = build_klifs_to_ca_index(row.get('klifs_residue_map', None), ca_data['residue_ids'])
    
    structures_data.append({
        'sample_id': row['sample_id'],
        'structure_id': row.get('structure_id', None),
        'pdb_id': pdb_code,
        'kinase_name': row['kinase_name'],
        'chain': row.get('chain', ca_data.get('chain_used', None)),
        'num_residues': ca_data['num_residues'],
        'ca_coords': ca_coords,
        'adjacency': adjacency,
        'num_edges': num_edges,
        'residues': ca_data['residues'],
        'residue_ids': ca_data['residue_ids'],
        'conformation_class': row['conformation_class'],
        'dfg_state': row.get('dfg_state', None),
        'alphac_state': row.get('alphac_state', None),
        'resolution': row.get('resolution', np.nan),
        'quality_score': row.get('quality_score', np.nan),
        'pocket': _clean_optional_value(row.get('pocket', None)),
        'klifs_residue_map': _parse_json_field(row.get('klifs_residue_map', None)),
        'klifs_to_ca_index': klifs_to_ca_index,
        'klifs_mapping_status': klifs_mapping_status,
    })

print(f"✓ Extracted {len(structures_data)} valid structures")
print(f"  PDB file errors: {ca_extraction_errors}")
print(f"  Invalid structures: {invalid_structures}")

# Statistics
if structures_data:
    num_residues = [s['num_residues'] for s in structures_data]
    num_edges = [s['num_edges'] for s in structures_data]
    mapped = sum(1 for s in structures_data if s.get('klifs_to_ca_index'))
    
    print(f"\nStructure statistics:")
    print(f"  Residues: min={min(num_residues)}, max={max(num_residues)}, mean={np.mean(num_residues):.1f}")
    print(f"  Edges: min={min(num_edges)}, max={max(num_edges)}, mean={np.mean(num_edges):.1f}")
    print(f"  KLIFS residue maps matched to Cα indices: {mapped}/{len(structures_data)}")


✓ Graph extraction functions defined

Extracting Cα coordinates from PDB files...

✓ Extracted 504 valid structures
  PDB file errors: 0
  Invalid structures: 291

Structure statistics:
  Residues: min=234, max=614, mean=293.5
  Edges: min=1986.0, max=5415.0, mean=2511.7
  KLIFS residue maps matched to Cα indices: 504/504


## 5. Build ESM Embeddings

In [7]:
def extract_sequence_from_pdb(pdb_file, preferred_chain=None):
    """Extract amino acid sequence from one PDB chain."""
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure('temp', str(pdb_file))
    
    # 3-letter to 1-letter amino acid mapping
    aa_map = {
        'ALA': 'A', 'ARG': 'R', 'ASN': 'N', 'ASP': 'D', 'CYS': 'C',
        'GLN': 'Q', 'GLU': 'E', 'GLY': 'G', 'HIS': 'H', 'ILE': 'I',
        'LEU': 'L', 'LYS': 'K', 'MET': 'M', 'PHE': 'F', 'PRO': 'P',
        'SER': 'S', 'THR': 'T', 'TRP': 'W', 'TYR': 'Y', 'VAL': 'V',
        'MSE': 'M',  # Selenomethionine
    }
    
    sequence = []
    selected_chain = str(preferred_chain).strip() if preferred_chain is not None and str(preferred_chain).strip() else None
    first_chain = None
    for model in structure:
        for chain in model:
            chain_id = chain.id.strip() or '_'
            if first_chain is None:
                first_chain = chain_id
            if selected_chain is None:
                selected_chain = first_chain
            if chain_id != selected_chain:
                continue
            for residue in chain:
                res_name = residue.get_resname().strip()
                aa = aa_map.get(res_name, 'X')
                sequence.append(aa)
    
    result = ''.join(sequence)
    if not result:
        raise ValueError(f"Could not extract sequence from {pdb_file}")
    return result

# Load ESM-2 model (33-layer, 650M parameters) - NO FALLBACK
print("Loading ESM-2 model (esm2_t33_650M_UR50D)...\n")
print("This is a large model (~1.3 GB). Download may take a few minutes on first run.\n")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}\n")

# Load model - MANDATORY, no try/except (fails loudly if not available)
model_name = "esm2_t33_650M_UR50D"
print(f"Loading {model_name} from Meta Hub...")
model, alphabet = esm.pretrained.load_model_and_alphabet_hub(model_name)

model.eval()
model = model.to(device)
print(f"✓ ESM-2 ({model_name}) loaded successfully on {device}")
print(f"  Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"  Output dimension: 1280\n")

def get_sequence_embedding_esm2(sequence, model, alphabet, device, repr_layer=33):
    """
    Get ESM-2 embedding for a sequence (NO FALLBACK VERSION).
    
    Args:
        sequence: amino acid sequence string
        model: ESM2 model instance
        alphabet: ESM2 alphabet
        device: 'cuda' or 'cpu'
        repr_layer: representation layer to extract (default: 33 for t33 model)
    
    Returns:
        np.array of shape (seq_len, 1280) - actual ESM-2 embeddings
    
    Raises:
        Exception if anything fails (no degradation)
    """
    batch_converter = alphabet.get_batch_converter()
    batch_labels, batch_strs, batch_tokens = batch_converter([("protein", sequence)])
    batch_tokens = batch_tokens.to(device)
    
    with torch.no_grad():
        results = model(batch_tokens, repr_layers=[repr_layer])
    
    embeddings = results["representations"][repr_layer]
    # Remove BOS and EOS special tokens, keep only actual residue embeddings
    embeddings = embeddings[0, 1:-1, :].cpu().numpy()
    
    return embeddings.astype(np.float32)

# Extract sequences and compute ESM-2 embeddings
print("=" * 70)
print("Computing ESM-2 embeddings for all structures")
print("=" * 70)
print()

sequence_cache = {}
embeddings_cache = {}

for i, struct in enumerate(structures_data):
    pdb_file = PDB_DIR / f"{struct['pdb_id']}.pdb"
    
    cache_key = (str(pdb_file), struct.get('chain'))
    if cache_key not in sequence_cache:
        # Extract sequence from the same chain used for Cα coordinates
        seq = extract_sequence_from_pdb(pdb_file, preferred_chain=struct.get('chain'))
        sequence_cache[cache_key] = seq
        
        # Compute ESM-2 embedding
        emb = get_sequence_embedding_esm2(seq, model=model, alphabet=alphabet, device=device, repr_layer=33)
        embeddings_cache[cache_key] = emb
        
        struct['sequence'] = seq
        struct['embedding'] = emb
    else:
        struct['sequence'] = sequence_cache[cache_key]
        struct['embedding'] = embeddings_cache[cache_key]
    
    if (i + 1) % 50 == 0:
        print(f"  [{i+1:4d}/{len(structures_data)}] Processed - Last: {struct['sample_id']}")

print(f"\n✓ ESM-2 embeddings computed for all {len(structures_data)} structures")

# Verify embeddings
first_emb = structures_data[0]['embedding']
print(f"\nEmbedding verification:")
print(f"  Shape: {first_emb.shape}")
print(f"  Dtype: {first_emb.dtype}")
print(f"  Min: {first_emb.min():.6f}, Max: {first_emb.max():.6f}")
print(f"  Mean: {first_emb.mean():.6f}, Std: {first_emb.std():.6f}")
print(f"  Sequence length: {len(structures_data[0]['sequence'])}")

# Sanity check: all embeddings must have shape (seq_len, 1280)
embedding_shapes = [s['embedding'].shape for s in structures_data]
if len(set(str(shape) for shape in embedding_shapes)) > len(structures_data) / 10:
    print("\n⚠ WARNING: Embedding shapes vary significantly!")
    shape_counts = {}
    for shape in embedding_shapes:
        shape_counts[str(shape)] = shape_counts.get(str(shape), 0) + 1
    for shape, count in sorted(shape_counts.items(), key=lambda x: -x[1])[:5]:
        print(f"    {shape}: {count} structures")
else:
    print(f"\n✓ All embeddings have consistent shape: {first_emb.shape}")

Loading ESM-2 model (esm2_t33_650M_UR50D)...

This is a large model (~1.3 GB). Download may take a few minutes on first run.

Using device: cpu

Loading esm2_t33_650M_UR50D from Meta Hub...
Downloading: "https://dl.fbaipublicfiles.com/fair-esm/models/esm2_t33_650M_UR50D.pt" to /Users/josefinadehan/.cache/torch/hub/checkpoints/esm2_t33_650M_UR50D.pt
Downloading: "https://dl.fbaipublicfiles.com/fair-esm/regression/esm2_t33_650M_UR50D-contact-regression.pt" to /Users/josefinadehan/.cache/torch/hub/checkpoints/esm2_t33_650M_UR50D-contact-regression.pt
✓ ESM-2 (esm2_t33_650M_UR50D) loaded successfully on cpu
  Model parameters: 651,043,254
  Output dimension: 1280

Computing ESM-2 embeddings for all structures

  [  50/504] Processed - Last: EGFR_5cno_A
  [ 100/504] Processed - Last: EGFR_5gmp_A
  [ 150/504] Processed - Last: EGFR_6tfw_A
  [ 200/504] Processed - Last: EGFR_8g63_A
  [ 250/504] Processed - Last: EGFR_7si1_A
  [ 300/504] Processed - Last: BRAF_5ct7_B
  [ 350/504] Processed - L

## 6. Final Validation & Dataset Readiness

In [8]:
print("="*70)
print("FINAL DATASET VALIDATION")
print("="*70)

# 1. Dataset size
print(f"\n1. Dataset Size:")
print(f"   Total samples: {len(structures_data)}")

# 2. Class distribution
class_dist = {}
for s in structures_data:
    cls = s['conformation_class']
    class_dist[cls] = class_dist.get(cls, 0) + 1

print(f"\n2. Conformation Class Distribution:")
for cls in sorted(class_dist.keys()):
    count = class_dist[cls]
    pct = 100 * count / len(structures_data)
    print(f"   {cls:10s}: {count:5d} ({pct:5.1f}%)")

# 3. Kinase distribution
kinase_dist = {}
for s in structures_data:
    kinase = s['kinase_name']
    kinase_dist[kinase] = kinase_dist.get(kinase, 0) + 1

print(f"\n3. Kinase Distribution:")
for kinase in sorted(kinase_dist.keys()):
    count = kinase_dist[kinase]
    print(f"   {kinase:10s}: {count:5d}")

# 4. Graph statistics
print(f"\n4. Graph Statistics:")
residues = [s['num_residues'] for s in structures_data]
edges = [s['num_edges'] for s in structures_data]

print(f"   Residues per structure:")
print(f"      Min: {min(residues)}, Max: {max(residues)}, Mean: {np.mean(residues):.1f} ± {np.std(residues):.1f}")
print(f"   Edges per structure:")
print(f"      Min: {min(edges)}, Max: {max(edges)}, Mean: {np.mean(edges):.1f} ± {np.std(edges):.1f}")

# 5. Embedding statistics
valid_embeddings = [s for s in structures_data if s['embedding'] is not None]
print(f"\n5. Embedding Statistics:")
print(f"   Valid embeddings: {len(valid_embeddings)}/{len(structures_data)}")

if valid_embeddings:
    emb_dims = [s['embedding'].shape[1] for s in valid_embeddings]
    emb_lens = [s['embedding'].shape[0] for s in valid_embeddings]
    print(f"   Embedding dimension: {emb_dims[0]}")
    print(f"   Sequence lengths: Min={min(emb_lens)}, Max={max(emb_lens)}, Mean={np.mean(emb_lens):.1f}")

# 6. Data integrity
print(f"\n6. Data Integrity Checks:")
checks = {
    'All have sample_id': all('sample_id' in s for s in structures_data),
    'All have CA coords': all(s['ca_coords'] is not None and len(s['ca_coords']) > 0 for s in structures_data),
    'All have adjacency': all(s['adjacency'] is not None for s in structures_data),
    'All have sequence': all(s['sequence'] is not None for s in structures_data),
    'All have conformation class': all(s['conformation_class'] in ['ACTIVE', 'INACTIVE', 'UNKNOWN'] for s in structures_data),
}

for check, result in checks.items():
    status = "✓" if result else "✗"
    print(f"   {status} {check}")

# 7. Ready for training?
all_valid = all(checks.values())
print(f"\n7. Ready for Model Training:")
if all_valid and len(structures_data) > 0 and len(valid_embeddings) > 0:
    print(f"   ✓ YES - Dataset is ready for EGNN/XGBoost")
    print(f"   ✓ {len(structures_data)} unique samples")
    print(f"   ✓ Graph representations ready")
    print(f"   ✓ ESM embeddings ready")
    print(f"   ✓ Class labels ready")
else:
    print(f"   ✗ NO - Dataset has issues")

print("\n" + "="*70)

FINAL DATASET VALIDATION

1. Dataset Size:
   Total samples: 504

2. Conformation Class Distribution:
   ACTIVE    :   261 ( 51.8%)
   INACTIVE  :   242 ( 48.0%)
   UNKNOWN   :     1 (  0.2%)

3. Kinase Distribution:
   ABL1      :    25
   BRAF      :   109
   EGFR      :   265
   FGFR1     :    61
   KIT       :    34
   PDGFRA    :    10

4. Graph Statistics:
   Residues per structure:
      Min: 234, Max: 614, Mean: 293.5 ± 44.1
   Edges per structure:
      Min: 1986.0, Max: 5415.0, Mean: 2511.7 ± 382.7

5. Embedding Statistics:
   Valid embeddings: 504/504
   Embedding dimension: 1280
   Sequence lengths: Min=248, Max=738, Mean=361.6

6. Data Integrity Checks:
   ✓ All have sample_id
   ✓ All have CA coords
   ✓ All have adjacency
   ✓ All have sequence
   ✓ All have conformation class

7. Ready for Model Training:
   ✓ YES - Dataset is ready for EGNN/XGBoost
   ✓ 504 unique samples
   ✓ Graph representations ready
   ✓ ESM embeddings ready
   ✓ Class labels ready



In [9]:
import pickle

DATASET_PKL = BASE_DIR / "dataset_ready.pkl"

# Serialize to pickle for fast loading in next notebooks
dataset_dict = {
    'structures': structures_data,
    'num_samples': len(structures_data),
    'class_distribution': class_dist,
    'kinase_distribution': kinase_dist,
}

with open(DATASET_PKL, 'wb') as f:
    pickle.dump(dataset_dict, f)

print(f"✓ Dataset saved to {DATASET_PKL}")
print(f"  File size: {DATASET_PKL.stat().st_size / (1024**2):.2f} MB")

# Also create a metadata CSV for reference
metadata = []
for s in structures_data:
    metadata.append({
        'sample_id': s['sample_id'],
        'pdb_id': s['pdb_id'],
        'structure_id': s.get('structure_id'),
        'chain': s.get('chain'),
        'kinase_name': s['kinase_name'],
        'num_residues': s['num_residues'],
        'num_edges': s['num_edges'],
        'conformation_class': s['conformation_class'],
        'dfg_state': s.get('dfg_state'),
        'alphac_state': s.get('alphac_state'),
        'pocket': s.get('pocket'),
        'klifs_mapping_status': s.get('klifs_mapping_status'),
        'klifs_to_ca_index': json.dumps(s.get('klifs_to_ca_index'), ensure_ascii=False) if s.get('klifs_to_ca_index') else None,
        'seq_length': len(s['sequence']) if s['sequence'] else 0,
        'embedding_dim': s['embedding'].shape[1] if s['embedding'] is not None else 0,
    })

df_metadata = pd.DataFrame(metadata)
METADATA_CSV = BASE_DIR / "dataset_metadata.csv"
df_metadata.to_csv(METADATA_CSV, index=False)

print(f"✓ Metadata saved to {METADATA_CSV}")
print(f"\nMetadata preview:")
print(df_metadata.head(3))

✓ Dataset saved to /Users/josefinadehan/vision_avanzada_t/vision_avanzada_t/dataset_ready.pkl
  File size: 1068.68 MB
✓ Metadata saved to /Users/josefinadehan/vision_avanzada_t/vision_avanzada_t/dataset_metadata.csv

Metadata preview:
     sample_id pdb_id  structure_id chain kinase_name  num_residues  \
0  EGFR_3w33_A   3w33           782     A        EGFR           297   
1  EGFR_2ito_A   2ito           783     A        EGFR           303   
2  EGFR_4li5_A   4li5           797     A        EGFR           306   

   num_edges conformation_class dfg_state alphac_state  \
0     2568.0           INACTIVE        in          out   
1     2576.0             ACTIVE        in           in   
2     2648.0             ACTIVE        in           in   

                                              pocket  \
0  KVLGSGAFGTVYKVAIKELEILDEAYVMASVDPHVCRLLGIQLITQ...   
1  KVLSSGAFGTVYKVAIKELEILDEAYVMASVDPHVCRLLGIQLITQ...   
2  KVLGSGAFGTVYKVAIKELEILDEAYVMASVDPHVCRLLGIQLITQ...   

        klifs_mapping_

In [10]:
print("="*80)
print("DETAILED INSPECTION OF dataset_ready.pkl")
print("="*80)

import pickle

with open(DATASET_PKL, 'rb') as f:
    dataset = pickle.load(f)

print(f"\n1. STRUCTURE:")
print(f"   Type: {type(dataset)}")
print(f"   Top-level keys: {list(dataset.keys())}")

print(f"\n2. SAMPLES:")
samples = dataset['structures']
print(f"   Number of samples: {len(samples)}")
print(f"   First sample keys: {list(samples[0].keys())}")

# Inspect first sample in detail
sample1 = samples[0]
print(f"\n3. FIRST SAMPLE DETAILS:")
print(f"   sample_id: {sample1['sample_id']}")
print(f"   kinase_name: {sample1['kinase_name']}")
print(f"   pdb_id: {sample1['pdb_id']}")
print(f"   conformation_class: {sample1['conformation_class']}")
print(f"   ca_coords shape: {sample1['ca_coords'].shape}")
print(f"   adjacency shape: {sample1['adjacency'].shape}")
print(f"   embedding shape: {sample1['embedding'].shape}")
print(f"   embedding dtype: {sample1['embedding'].dtype}")

# Check embedding values
emb = sample1['embedding']
print(f"\n4. EMBEDDING STATISTICS (first sample):")
print(f"   Min value: {emb.min():.6f}")
print(f"   Max value: {emb.max():.6f}")
print(f"   Mean: {emb.mean():.6f}")
print(f"   Std: {emb.std():.6f}")
print(f"   First 10 values of first residue: {emb[0, :10]}")

# Check all embeddings
print(f"\n5. EMBEDDING DIMENSIONS ACROSS ALL SAMPLES:")
emb_dims = set()
for s in samples:
    emb_dims.add(s['embedding'].shape[1])
print(f"   Unique embedding dimensions: {emb_dims}")

# Conformation distribution
print(f"\n6. CONFORMATION DISTRIBUTION:")
conf_counts = {}
for s in samples:
    conf = s['conformation_class']
    conf_counts[conf] = conf_counts.get(conf, 0) + 1
for conf, count in sorted(conf_counts.items()):
    pct = 100 * count / len(samples)
    print(f"   {conf:12s}: {count:4d} ({pct:5.1f}%)")

# Kinase distribution
print(f"\n7. KINASE DISTRIBUTION:")
kinase_counts = {}
for s in samples:
    k = s['kinase_name']
    kinase_counts[k] = kinase_counts.get(k, 0) + 1
for kinase in sorted(kinase_counts.keys()):
    count = kinase_counts[kinase]
    pct = 100 * count / len(samples)
    print(f"   {kinase:10s}: {count:4d} ({pct:5.1f}%)")

print(f"\n8. STRUCTURE STATISTICS:")
residue_counts = [s['ca_coords'].shape[0] for s in samples]
edge_counts = [s['adjacency'].sum() // 2 for s in samples]  # Divide by 2 for undirected
print(f"   Residues per structure:")
print(f"     Min: {min(residue_counts)}, Max: {max(residue_counts)}, Mean: {np.mean(residue_counts):.1f}")
print(f"   Edges per structure:")
print(f"     Min: {min(edge_counts)}, Max: {max(edge_counts)}, Mean: {np.mean(edge_counts):.1f}")

print(f"\n{'='*80}")
print(f"✓ Dataset is fully loaded and ready for EGNN training")
print(f"{'='*80}")

DETAILED INSPECTION OF dataset_ready.pkl

1. STRUCTURE:
   Type: <class 'dict'>
   Top-level keys: ['structures', 'num_samples', 'class_distribution', 'kinase_distribution']

2. SAMPLES:
   Number of samples: 504
   First sample keys: ['sample_id', 'structure_id', 'pdb_id', 'kinase_name', 'chain', 'num_residues', 'ca_coords', 'adjacency', 'num_edges', 'residues', 'residue_ids', 'conformation_class', 'dfg_state', 'alphac_state', 'resolution', 'quality_score', 'pocket', 'klifs_residue_map', 'klifs_to_ca_index', 'klifs_mapping_status', 'sequence', 'embedding']

3. FIRST SAMPLE DETAILS:
   sample_id: EGFR_3w33_A
   kinase_name: EGFR
   pdb_id: 3w33
   conformation_class: INACTIVE
   ca_coords shape: (297, 3)
   adjacency shape: (297, 297)
   embedding shape: (426, 1280)
   embedding dtype: float32

4. EMBEDDING STATISTICS (first sample):
   Min value: -7.737173
   Max value: 3.424824
   Mean: -0.000132
   Std: 0.266733
   First 10 values of first residue: [ 0.02840171  0.02910328 -0.202895

## Dataset Reference: `dataset_ready.pkl`

### File Information
- **Location**: `/home/user/tpf2/vision_avanzada_tpf/dataset_ready.pkl`
- **Size**: ~1,149 MB
- **Format**: Python pickle

### Top-Level Structure
```python
dataset = {
    'structures': [...],           # List of 795 samples
    'num_samples': 795,            # Total count
    'class_distribution': {...},   # Conformation distribution
    'kinase_distribution': {...}   # Kinase distribution
}
```

### Per-Sample Schema
```python
sample = {
    'sample_id': 'EGFR_3w33_A',              # Unique ID
    'kinase_name': 'EGFR',                   # Kinase
    'pdb_id': '3w33',                        # PDB code
    'chain': 'A',                            # Chain ID
    'conformation_class': 'INACTIVE',        # Label
    'ca_coords': np.ndarray,                 # (297, 3) Cα positions
    'adjacency': np.ndarray,                 # (297, 297) distance graph
    'embedding': np.ndarray,                 # (426, 1280) ESM2 embeddings
    'sequence': 'MPKK...',                   # Amino acid sequence
}
```

### Key Statistics
- **Total Samples**: 795
- **ACTIVE**: 368 (46.3%) | **INACTIVE**: 424 (53.3%) | **UNKNOWN**: 3 (0.4%)
- **ESM2 Dimension**: 1280 per residue
- **Residues per structure**: 59–426 (mean: 261.8)

In [11]:
# Example: How to load and use the dataset

# Load dataset
import pickle
with open('dataset_ready.pkl', 'rb') as f:
    dataset = pickle.load(f)

# Access data
structures = dataset['structures']
print(f"Total samples: {dataset['num_samples']}")
print(f"Class distribution: {dataset['class_distribution']}")
print(f"Kinase distribution: {dataset['kinase_distribution']}")

# Access first sample
sample = structures[0]
print(f"\nSample ID: {sample['sample_id']}")
print(f"Cα coords shape: {sample['ca_coords'].shape}")
print(f"Adjacency shape: {sample['adjacency'].shape}")
print(f"Embedding shape: {sample['embedding'].shape}")
print(f"Label: {sample['conformation_class']}")

# For EGNN training: use (ca_coords, adjacency) as graph input
# For XGBoost: flatten embedding as features
print(f"\nReady for:")
print(f"  - EGNN: graph nodes (Cα coords) + adjacency matrix")
print(f"  - XGBoost: ESM2 embeddings (1280-dim per residue)")

Total samples: 504
Class distribution: {'INACTIVE': 242, 'ACTIVE': 261, 'UNKNOWN': 1}
Kinase distribution: {'EGFR': 265, 'BRAF': 109, 'ABL1': 25, 'KIT': 34, 'PDGFRA': 10, 'FGFR1': 61}

Sample ID: EGFR_3w33_A
Cα coords shape: (297, 3)
Adjacency shape: (297, 297)
Embedding shape: (426, 1280)
Label: INACTIVE

Ready for:
  - EGNN: graph nodes (Cα coords) + adjacency matrix
  - XGBoost: ESM2 embeddings (1280-dim per residue)
